# Canonical Review Cleaning Validation

This notebook explains and validates the reusable Amazon review preprocessing pipeline. It does not contain a second implementation of cleaning.

## Objectives

- demonstrate every cleaning rule on a small readable fixture;
- verify primary rejection reasons and exact-duplicate removal;
- validate deterministic review IDs, UTC timestamps, text normalization, and optional fields;
- reconcile the saved full-dataset report with the canonical Parquet schema.

# Проверка очистки отзывов

Этот ноутбук объясняет и проверяет переиспользуемый конвейер предобработки Amazon-отзывов. Здесь нет второй версии очистки: мы вызываем тот же код из `src/`, который используется для построения полного канонического датасета. Это снижает риск, что демонстрация в ноутбуке и реальная обработка начнут работать по-разному.

## Цели

- показать каждое правило очистки на маленьком и понятном примере;
- проверить основные причины отклонения строк и удаление точных дубликатов;
- проверить стабильные ID отзывов, время UTC, нормализацию текста и необязательные поля;
- сверить отчет полного запуска с канонической схемой Parquet.

## Inputs, outputs, and assumptions

The notebook reads the versioned dataset manifest, creates a temporary JSONL fixture, calls `src.preprocessing.reviews.preprocess_reviews`, and reads the existing production quality report and canonical Parquet metadata.

A *primary rejection reason* is the first failed mandatory rule in a fixed order. One rejected row therefore contributes to exactly one drop counter, which makes input/output reconciliation unambiguous.

## Входы, результаты и допущения

Ноутбук читает версионный манифест, создает временный тестовый JSONL-файл, вызывает `src.preprocessing.reviews.preprocess_reviews`, а затем читает существующий отчет полного запуска и метаданные канонического Parquet. Временный пример удаляется в конце и не изменяет рабочие данные проекта.

*Основная причина отклонения* — первое нарушенное обязательное правило в заранее заданном порядке. Поэтому отклоненная строка попадает ровно в один счетчик, и баланс входных и выходных строк можно проверить однозначно.

In [ ]:
# Standard library / Стандартная библиотека
import json
import sys
from dataclasses import asdict
from pathlib import Path
from tempfile import TemporaryDirectory

# Third-party packages / Сторонние библиотеки
import duckdb
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# Locate the repository root before importing local project modules.
# Находим корень репозитория до импорта локальных модулей проекта.
PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Local project modules / Локальные модули проекта
from src.common.project import find_project_root
from src.ingestion.dataset_manifest import load_dataset_manifest
from src.preprocessing.reviews import (
    CANONICAL_REVIEW_SCHEMA,
    ReviewPreprocessingConfig,
    preprocess_reviews,
    validate_canonical_review_parquet,
)

In [ ]:
# All persistent paths come from one dataset version.
# Все постоянные пути относятся к одной версии датасета.
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
MANIFEST_PATH = (
    PROJECT_ROOT
    / "config/datasets/amazon_reviews_2023_beauty_2021_2023_v1.json"
)
QUALITY_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/data_quality/amazon_reviews_2023_beauty_2021_2023_v1.json"
)

manifest = load_dataset_manifest(MANIFEST_PATH)
canonical_registration = manifest.file_by_role("canonical_reviews")
CANONICAL_REVIEWS_PATH = PROJECT_ROOT / canonical_registration.path

# A temporary directory keeps the demonstration isolated from real artifacts.
# Временная директория отделяет учебный пример от реальных артефактов.
temporary_directory = TemporaryDirectory(prefix="cleaning_validation_")
TEMP_DIR = Path(temporary_directory.name)
FIXTURE_SOURCE_PATH = TEMP_DIR / "reviews.jsonl"
FIXTURE_OUTPUT_PATH = TEMP_DIR / "canonical_reviews.parquet"
FIXTURE_REPORT_PATH = TEMP_DIR / "quality_report.json"

fixture_config = ReviewPreprocessingConfig(
    dataset_version="beauty_cleaning_fixture_v1",
    dataset_category=manifest.dataset_category,
    start_date=manifest.date_window.start,
    end_date=manifest.date_window.end,
    chunk_size=3,
    deduplicate=True,
    deduplication_memory_limit="512MB",
)

# Make the actual notebook environment visible before pipeline execution.
# Показываем реальное окружение ноутбука до запуска конвейера обработки.
print(f"Python executable: {sys.executable}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Production dataset: {manifest.dataset_version}")
print(f"Fixture window: {fixture_config.start_date} through {fixture_config.end_date}")

## 1. Readable cleaning fixture

The fixture deliberately contains valid rows, exact duplicates with both known and anonymous users, a title-only review, invalid mandatory values, malformed/fractional/overflowing timestamps, and malformed/fractional/overflowing optional vote values. This is an explanatory integration check; small edge-case regression tests also live in `tests/test_review_preprocessing.py`.

## Понятный тестовый пример

В примере специально собраны корректные строки, точные дубликаты с известным и анонимным пользователем, отзыв только с заголовком, ошибки обязательных полей, некорректные/дробные/переполненные timestamps и такие же пограничные значения голосов полезности. Это наглядная интеграционная проверка, а небольшие автоматические регрессионные тесты для пограничных случаев находятся в `tests/test_review_preprocessing.py`.

In [ ]:
first_review = {
    "rating": 5.0,
    "title": "  Great <b>conditioner</b>  ",
    "text": "Leaves   my hair soft.",
    "asin": "B000000001",
    "parent_asin": "B000000000",
    "user_id": "USER-1",
    "timestamp": 1672531200000,
    "helpful_vote": 2,
    "verified_purchase": True,
}
source_records = [
    first_review,
    first_review.copy(),
    {**first_review, "title": "Useful title", "text": "", "asin": "B000000002"},
    {**first_review, "rating": 7},
    {**first_review, "parent_asin": ""},
    {**first_review, "title": "", "text": ""},
    {**first_review, "timestamp": 1609459199000},
    {
        **first_review,
        "asin": "B000000003",
        "user_id": "USER-3",
        "timestamp": 1675209600000,
        "helpful_vote": -1,
        "verified_purchase": "yes",
    },
    {**first_review, "timestamp": "not-a-timestamp"},
    {**first_review, "timestamp": 1672531200000.5},
    {**first_review, "timestamp": 10**30},
    {
        **first_review,
        "title": "Anonymous review",
        "asin": "B000000004",
        "user_id": None,
        "timestamp": 1677628800000,
    },
    {
        **first_review,
        "title": "Anonymous review",
        "asin": "B000000004",
        "user_id": None,
        "timestamp": 1677628800000,
    },
    {
        **first_review,
        "title": "Malformed helpful vote",
        "asin": "B000000005",
        "user_id": "USER-5",
        "timestamp": 1680307200000,
        "helpful_vote": "many",
    },
    {
        **first_review,
        "title": "Fractional helpful vote",
        "asin": "B000000006",
        "user_id": "USER-6",
        "timestamp": 1682899200000,
        "helpful_vote": 1.5,
    },
    {
        **first_review,
        "title": "Overflowing helpful vote",
        "asin": "B000000007",
        "user_id": "USER-7",
        "timestamp": 1685577600000,
        "helpful_vote": 2147483648,
    },
]

# JSONL stores one independent JSON object per line.
# В JSONL каждая строка является отдельным JSON-объектом.
FIXTURE_SOURCE_PATH.write_text(
    "".join(json.dumps(record) + "\n" for record in source_records),
    encoding="utf-8",
)
display(pd.DataFrame(source_records))

In [ ]:
# Call the production pipeline; the notebook does not reproduce its rules.
# Вызываем рабочий конвейер обработки и не переписываем его правила в ноутбуке.
fixture_report = preprocess_reviews(
    FIXTURE_SOURCE_PATH,
    FIXTURE_OUTPUT_PATH,
    config=fixture_config,
    report_path=FIXTURE_REPORT_PATH,
)
display(pd.Series(asdict(fixture_report), name="value").to_frame())

## 2. Row reconciliation

Every input row must become either a valid pre-deduplication row or one primary rejection. Every valid row must then become either a retained canonical row or an exact duplicate. These two equalities detect silent row loss.

## Баланс строк

Каждая входная строка должна стать либо корректной строкой до дедупликации, либо получить одну основную причину отклонения. Затем каждая корректная строка должна стать либо сохраненным каноническим отзывом, либо точным дубликатом. Два равенства ниже обнаруживают незаметную потерю строк между этапами.

In [ ]:
expected_drop_reasons = {
    "empty_review_text": 1,
    "invalid_rating": 1,
    "invalid_timestamp": 3,
    "missing_parent_asin": 1,
    "outside_date_window": 1,
}
input_reconciles = fixture_report.input_rows == (
    fixture_report.valid_rows_before_deduplication
    + sum(fixture_report.dropped_by_primary_reason.values())
)
deduplication_reconciles = fixture_report.valid_rows_before_deduplication == (
    fixture_report.output_rows + fixture_report.duplicate_rows_removed
)

assert fixture_report.input_rows == 16
assert fixture_report.dropped_by_primary_reason == expected_drop_reasons
assert fixture_report.valid_rows_before_deduplication == 9
assert fixture_report.duplicate_rows_removed == 2
assert fixture_report.output_rows == 7
assert fixture_report.invalid_optional_values == {
    "helpful_vote": 4,
    "verified_purchase": 1,
}
assert input_reconciles and deduplication_reconciles

display(
    pd.DataFrame(
        {
            "check": ["input reconciliation", "deduplication reconciliation"],
            "passed": [input_reconciles, deduplication_reconciles],
        }
    )
)

## 3. Canonical transformations

Mandatory invalid values cause rejection. Malformed, fractional, or overflowing numeric values are classified deterministically instead of crashing conversion. Invalid optional values become null because the review can still be useful without them. Title and body are cleaned separately and combined into `review_text`; therefore a title-only review remains valid. Exact duplicate identity works for both known and null `user_id` values, while stable `review_id` values use dataset version plus immutable source-row position.

## Канонические преобразования

Ошибка обязательного поля приводит к отклонению строки. Некорректные, дробные и переполненные числовые значения получают детерминированный результат вместо аварийного завершения преобразования. Ошибка необязательного поля превращается в null, потому что сам отзыв все еще полезен. Заголовок и текст очищаются отдельно и объединяются в `review_text`, поэтому отзыв только с заголовком сохраняется. Точная дедупликация работает и при null в `user_id`, а стабильный `review_id` строится из версии датасета и неизменной позиции строки в исходном файле.

In [ ]:
fixture_parquet = pq.ParquetFile(FIXTURE_OUTPUT_PATH)
fixture_reviews = pd.read_parquet(FIXTURE_OUTPUT_PATH)
fixture_validation = validate_canonical_review_parquet(FIXTURE_OUTPUT_PATH)

assert fixture_validation.is_valid
assert fixture_reviews["source_record_index"].tolist() == [
    0, 2, 7, 11, 13, 14, 15
]
assert fixture_reviews["review_id"].is_unique
assert fixture_reviews.loc[0, "review_text"] == (
    "Great conditioner Leaves my hair soft."
)
assert fixture_reviews.loc[1, "review_text"] == "Useful title"
assert pd.isna(fixture_reviews.loc[2, "helpful_vote"])
assert pd.isna(fixture_reviews.loc[2, "verified_purchase"])
assert pd.isna(fixture_reviews.loc[3, "user_id"])
assert fixture_reviews.loc[3, "review_text"].startswith("Anonymous review")
assert fixture_reviews.loc[4:, "helpful_vote"].isna().all()

display(
    fixture_reviews[
        [
            "review_id",
            "source_record_index",
            "asin",
            "review_text",
            "review_timestamp",
            "helpful_vote",
            "verified_purchase",
        ]
    ]
)

## 4. Full production artifact

The small fixture proves understandable behavior. The next check confirms that the saved full Beauty build has the same schema, matches its registered row count, and has a reconciled quality report. It reads Parquet metadata rather than loading millions of reviews into notebook memory.

## Полный рабочий артефакт

Маленький пример делает поведение понятным. Следующая проверка подтверждает, что полный Beauty-датасет использует ту же схему, совпадает с зарегистрированным количеством строк и имеет согласованный отчет качества. Мы читаем метаданные Parquet, а не загружаем миллионы отзывов в память ноутбука.

In [ ]:
if not QUALITY_REPORT_PATH.is_file():
    raise FileNotFoundError(QUALITY_REPORT_PATH)
if not CANONICAL_REVIEWS_PATH.is_file():
    raise FileNotFoundError(CANONICAL_REVIEWS_PATH)

production_report = json.loads(QUALITY_REPORT_PATH.read_text(encoding="utf-8"))
production_parquet = pq.ParquetFile(CANONICAL_REVIEWS_PATH)
production_validation = validate_canonical_review_parquet(
    CANONICAL_REVIEWS_PATH
)
production_input_reconciles = production_report["input_rows"] == (
    production_report["valid_rows_before_deduplication"]
    + sum(production_report["dropped_by_primary_reason"].values())
)
production_deduplication_reconciles = (
    production_report["valid_rows_before_deduplication"]
    == production_report["output_rows"]
    + production_report["duplicate_rows_removed"]
)

assert production_input_reconciles
assert production_deduplication_reconciles
assert production_report["dataset_version"] == manifest.dataset_version
assert production_validation.row_count == canonical_registration.record_count
assert production_validation.is_valid

production_summary = pd.Series(
    {
        "input_rows": production_report["input_rows"],
        "valid_rows_before_deduplication": production_report["valid_rows_before_deduplication"],
        "duplicate_rows_removed": production_report["duplicate_rows_removed"],
        "output_rows": production_report["output_rows"],
        "parquet_row_groups": production_parquet.metadata.num_row_groups,
        "input_reconciliation_passed": production_input_reconciles,
        "deduplication_reconciliation_passed": production_deduplication_reconciles,
        "canonical_schema_passed": production_validation.is_valid,
    },
    name="value",
).to_frame()
schema_table = pd.DataFrame(
    {
        "column": CANONICAL_REVIEW_SCHEMA.names,
        "type": [str(field.type) for field in CANONICAL_REVIEW_SCHEMA],
        "contract_nullable": [field.nullable for field in CANONICAL_REVIEW_SCHEMA],
        "physical_nullable": [field.nullable for field in production_parquet.schema_arrow],
        "actual_null_count": [
            production_validation.required_null_counts.get(field.name)
            for field in CANONICAL_REVIEW_SCHEMA
        ],
    }
)
display(production_summary)
display(schema_table)

In [ ]:
# Explicit cleanup makes the notebook's temporary side effect visible.
# Явное удаление показывает, что временные файлы не являются артефактами проекта.
temporary_directory.cleanup()
print("Temporary validation files removed.")

## 5. Conclusion and limitations

The reusable pipeline passed the explanatory fixture checks, including bounded numeric coercion and anonymous exact deduplication, and the saved full artifact passed row reconciliation plus strict type/nullability validation. The canonical output preserves deterministic source order after removing exact duplicates and records every rejected row under one primary reason.

This notebook does not prove that the filtered source itself contains every relevant raw Amazon review, nor does it validate product-metadata joins or category-path coverage. Those are separate source/catalog checks. Exact deduplication also does not remove paraphrases or repeated promotional text; near-duplicate analysis belongs to a later quality stage.

## Вывод и ограничения

Переиспользуемый конвейер прошел проверки на учебном примере, включая безопасное ограничение числовых значений и точную дедупликацию анонимных отзывов, а полный артефакт прошел сверку строк и строгую проверку типов/nullability. Канонический результат сохраняет детерминированный порядок исходных строк после удаления точных дубликатов и фиксирует каждую отклоненную строку ровно одной основной причиной.

Этот ноутбук не доказывает, что промежуточный отфильтрованный файл содержит все нужные исходные Amazon-отзывы, и не проверяет соединение с товарами или полноту категорий. Это отдельные проверки источника и каталога. Точная дедупликация также не удаляет перефразированные и почти одинаковые повторы — их анализ относится к следующему этапу качества данных.